In [0]:
#------------------------- Permite ejecutar query SQL mediante el sql_safe -------------------------
def sqlsafe(query):
  try:
    print ("sql_safe: query -> " + query)
    return spark.sql(query)
  except Exception as e:
    raise Exception(f'{{"coderror":"20001", "msgerror":"Error grave procesando consulta -> {str(e)}"}}')

In [0]:
#------------------------- Genera el ultimo dia habil del mes -------------------------
def ultimo_dia_habil(bd, aaaa_mm):
  try:
    ultimo_dia = """ select max(cal_ano || string(LPAD(TRIM(cast(cal_mes as char(11))),2,'0')) ||  string(LPAD(TRIM(cast(cal_dia as char(11))),2,'0'))) as periodo_id 
                       FROM  """ +bd +""".bci_cal_dia 
                      where cal_ano || string(LPAD(TRIM(cast(cal_mes as char(11))),2,'0')) = '""" +str(aaaa_mm)+"""' 
                        and TRIM(cal_ind_dia) = 'H' 
                        and fecha_proceso = (select max(fecha_proceso) from """ +bd +""".bci_cal_dia where substring(fecha_proceso, 1, 6) = '""" +str(aaaa_mm)+"""') """
    return sql_safe(ultimo_dia)
  except Exception as e:
    dbutils.notebook.exit("{\"coderror\":\"20001\", \"msgerror\":\"Error grave computando auxiliar -> "+str(e)+"\"}")

In [0]:
#------------------------- Genera el ultimo dia del mes -------------------------
def ultimo_dia_mes(bd, aaaa_mm):
  try:
    ultimo_dia = """ select max(string(LPAD(TRIM(cast(cal_dia as char(11))),2,'0')))   as max_dia_mes
                     from """ +bd +""".bci_cal_dia
                     where concat(cal_ano, string(LPAD(TRIM(cast(cal_mes as char(11))),2,'0'))) = '""" +str(aaaa_mm)+"""'  """
    return sql_safe(ultimo_dia)
  except Exception as e:
    dbutils.notebook.exit("{\"coderror\":\"20001\", \"msgerror\":\"Error grave computando auxiliar -> "+str(e)+"\"}")

In [0]:
#------------------------- Genera el formato del ultimo dia habil del mes -------------------------
def ultimo_dia_habil_format(bd, aaaa_mm):
  try:
    ultimo_dia_format = """ select max(cal_ano || '-' || string(LPAD(TRIM(cast(cal_mes as char(11))),2,'0')) || '-' ||  string(LPAD(TRIM(cast(cal_dia as char(11))),2,'0'))) as periodo_id 
                       FROM  """ +bd +""".bci_cal_dia 
                      where cal_ano || string(LPAD(TRIM(cast(cal_mes as char(11))),2,'0')) = '""" +str(aaaa_mm)+"""' 
                        and TRIM(cal_ind_dia) = 'H'
                        and fecha_proceso = (select max(fecha_proceso) from dsr_slv_Parametricas_db.bci_cal_dia where substring(fecha_proceso, 1, 6) = '""" +str(aaaa_mm)+"""') """
    return sql_safe(ultimo_dia_format)
  except Exception as e:
    dbutils.notebook.exit("{\"coderror\":\"20001\", \"msgerror\":\"Error grave computando auxiliar -> "+str(e)+"\"}")

In [0]:
#------------------------- Genera el periodo con 12 meses atras de la fecha pasada -------------------------
def periodo_anterior(diaHabil_periodo):
  try:
    fec_periodo_anterior_format = """ select add_months('"""+str(diaHabil_periodo)+"""', -12)  as fecha """
    return sql_safe(fec_periodo_anterior_format)
  except Exception as e:
    dbutils.notebook.exit("{\"coderror\":\"20001\", \"msgerror\":\"Error grave computando auxiliar -> "+str(e)+"\"}")

In [0]:
#------------------------- Encargada de ejecutar query SQL -------------------------
def sql_safe(query):
  try:
    print ("sql_safe: query -> " + query)
    return spark.sql(query)
  except Exception as e:
    raise Exception(f'{{"coderror":"20001", "msgerror":"Error grave procesando consulta -> {str(e)}"}}')
    
def sql_create_temp_view(name, query):
  try:
    print ("sql_create_temp_view: name ->" + name + ", query -> " + query)
    return sql_safe(query).cache.createOrReplaceTempView(name)
  except Exception as e:
    dbutils.notebook.exit("{\"coderror\":\"20001\", \"msgerror\":\"Error grave creando temp view -> "+str(e)+"\"}")
    
def sql_drop_cache_table(name):
  try:
    print ("sql_drop_cache_table: name ->" + name)
    return spark.sql("drop table if EXISTS " + name)
  except Exception as e:
    dbutils.notebook.exit("{\"coderror\":\"20001\", \"msgerror\":\"Error grave eliminando tabla cache -> "+str(e)+"\"}")
    
def sql_create_cache_table(name, query):
  try:
    print ("sql_create_cache_table: name ->" + name + ", query -> " + query)
    return spark.sql("cache table " + name + " " + query)
  except Exception as e:
    dbutils.notebook.exit("{\"coderror\":\"20001\", \"msgerror\":\"Error grave creando tabla cache -> "+str(e)+"\"}")

In [0]:
def valida_bd(nombrebdx, nombre):
    catalog, schema = nombrebdx.split(".", 1)

    schemas = spark.sql(f"SHOW SCHEMAS IN {catalog} LIKE '{schema}'").collect()

    if len(schemas) == 0:
        raise Exception(f'{{"coderror":28009, "msgerror":"Base de Datos no encontrada: {nombrebdx}"}}')

    print(f"Parametro {nombre}: {nombrebdx} ---> Base de Datos encontrada")

In [0]:
#------------------------- Valida parametros no vengan vacíos -------------------------
#Librerias
import json
from datetime import datetime, date, time, timedelta
from dateutil.relativedelta import relativedelta

##Hora de inicio
hora_ini = datetime.now()

#Funcion Control Parametros Vacíos
def valida_param_vacio(valor, nombre):
  if (len(valor) == 0):
    raise Exception(f'{{"coderror": 24001, "msgerror": "Falta parámetro: {nombre}", "descerror": "Falta parámetro {nombre}" }}')
  else:
    print(f"""Parametro {nombre}: {valor}""")

In [0]:
#------------------------- Valida que la fecha sea numerica -------------------------
def es_numerica(valor, nombre):

  if not valor.isnumeric():
    raise Exception(f'{{"coderror": 24001, "msgerror": " parámetro: {nombre}", "descerror": " error no es nro {nombre}" }}') 
  else:
    print(f"""Parametro {nombre} es nro: {valor}""")

In [0]:
#------------------------- Valida que la fecha sea valida -------------------------
def valida_fecha_valida(periodox, nombre):
  try:
    if len(periodox) == 10:
      periodo = datetime.strptime(periodox, '%Y-%m-%d')
      print(f"""Parametro {nombre}: {periodox} ---> Fecha valida""")
    elif len(periodox) == 8:
      periodo = datetime.strptime(periodox, '%Y%m%d')
      print(f"""Parametro {nombre}: {periodox} ---> Fecha valida""")
    elif len(periodox) == 6:
      periodo = datetime.strptime(periodox, '%Y%m')
      print(f"""Parametro {nombre}: {periodox} ---> Fecha valida""")
    else:
      raise Exception("{\"coderror\":28005, \"msgerror\":\"Largo de parámetro de entrada período inválido: "+periodox+"\"}")
  except ValueError:
    raise Exception("{\"coderror\":28006, \"msgerror\":\"Parámetro de entrada período inválido: "+periodox+"\"}")

In [0]:

#------------------------- Valida que la fecha no sea posterior a la actual -------------------------
#Librerias
import json
from datetime import datetime, date, time, timedelta
from dateutil.relativedelta import relativedelta

# Obtener fecha actual
current_date_str = datetime.now().strftime("%Y-%m-%d")
current_date = datetime.strptime(current_date_str, "%Y-%m-%d")
formatted_date = str(current_date.strftime("%Y%m%d"))

def valida_fecha_futura(periodox, nombre):
  try:
    if periodox > formatted_date:
      raise Exception("{\"coderror\":28000, \"msgerror\":\"Parámetro de entrada período no puede ser superior a fecha actual: "+periodox+"\"}")
    else:
      print(f"""Parametro {nombre}: {periodox} ---> Periodo no es superior a la fecha actual""")
  except ValueError:
    raise Exception("{\"coderror\":28000, \"msgerror\":\"Parámetro de entrada período no puede ser superior a fecha actual: "+periodox+"\"}")

In [0]:
#------------------------- Valida que la ruta exista -------------------------
def valida_ruta(valor, nombre):
  try:
    ruta = dbutils.fs.ls(f"{valor}")
    print(f"""Parametro {nombre}: {valor} ---> Ruta encontrada""")
  except ValueError:
      raise Exception("{\"coderror\":28010, \"msgerror\":\"Ruta no encontrada: "+str(valor)+"\"}")